In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

In [7]:
df = pd.read_csv("../data/processed_reviews.csv")

y = df["sentiment"]

tf = TfidfVectorizer(max_features=10000)
X = tf.fit_transform(df["review"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN Layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # Fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        
        output, _ = self.rnn(x, h0)

        output = self.fc(output[:, -1, :])
        return output

In [9]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())